# TRAINING

In [1]:
import pandas as pd


import sys
import os

from pathlib import Path
from itertools import islice
from torch.utils.data import random_split
sys.path.append(os.path.abspath('../src'))

In [3]:
from torch.utils.data import DataLoader
import torch.optim as optim
from torch.utils.data import random_split
from sentiment_model import SentimentModel
import torch.nn as nn
import torch


In [4]:
from tokenizer import Tokenizer
from prep_sentiment_data import prepare_data,save_split

In [5]:
df = pd.read_csv('./../data/IMDB Dataset.csv')

df.head()


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [6]:
processed_data = prepare_data("./../data/IMDB Dataset.csv")



The shape of row is :(2,)
The row content is: I really liked this Summerslam due to the look of the arena, the curtains and just the look overall was interesting to me for some reason. Anyways, this could have been one of the best Summerslam's ever if the WWF didn't have Lex Luger in the main event against Yokozuna, now for it's time it was ok to have a huge fat man vs a strong man but I'm glad times have changed. It was a terrible main event just like every match Luger is in is terrible. Other matches on the card were Razor Ramon vs Ted Dibiase, Steiner Brothers vs Heavenly Bodies, Shawn Michaels vs Curt Hening, this was the event where Shawn named his big monster of a body guard Diesel, IRS vs 1-2-3 Kid, Bret Hart first takes on Doink then takes on Jerry Lawler and stuff with the Harts and Lawler was always very interesting, then Ludvig Borga destroyed Marty Jannetty, Undertaker took on Giant Gonzalez in another terrible match, The Smoking Gunns and Tatanka took on Bam Bam Bigelow an

In [6]:
# This will show how many 'positive' and 'negative' rows exist
print(df['sentiment'].value_counts())

sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [6]:
print(len(processed_data))

50000


### Splitting Dataset

In [7]:
total_length = len(processed_data)
train_dataset = save_split(processed_data[:int(0.8*total_length)],"./../data/train_dataset")
validation_dataset = save_split(processed_data[int(0.8*total_length):int(0.9*total_length)],"./../data/validation_dataset")
test_dataset= save_split(processed_data[int(0.9*total_length):],"./../data/test_dataset")

In [7]:
# Let's say we have 25,000 reviews
train_size = int(0.8 * len(processed_data)) # 80% for training
val_size = len(processed_data) - train_size  # 20% for validation
train_subset ,val_subset,test_subset = random_split(processed_data,[train_size,val_size//2,val_size//2])

In [8]:
model = SentimentModel(vocab_size=10000,embedding_dim=100)

#The Judge
criterion = nn.CrossEntropyLoss()


#The Coach
optimizer = optim.Adam(model.parameters(),lr=0.001,weight_decay=1e-5)

schedular = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='min',factor=0.1,patience=3)

train_loader = DataLoader(dataset=train_subset,batch_size=32,shuffle=True)

val_loader  = DataLoader(dataset=val_subset,batch_size=32,shuffle=True)

In [9]:
epochs = 20

best_val_loss = float('inf')

patience_counter = 0


early_stop_patience = 5


for epoch in range(epochs):
    
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for text, labels in train_loader:
        optimizer.zero_grad()
        
        
        # 1. Combine the list of tensors into one 2D tensor: [Batch Size, 250]
        # We stack them so the model sees a single block of data
        text = torch.stack(text).long()
        #print(text.shape)
        text = text.transpose(0, 1)
        # 2. Ensure labels are a 1D tensor of integers
        # CrossEntropyLoss requires the 'long' data type for category indices
        labels = torch.as_tensor(labels).long()
        
        
        outputs = model(text)
        
        loss = criterion(outputs,labels)
        
        
        loss.backward()
        
        optimizer.step()
        
        running_loss += loss.item()
        
        #pick the highest score as the prediction
        _,predicted = torch.max(outputs.data,1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    epoch_loss = running_loss / len(train_loader)
    train_accuracy = 100 * correct / total
    
    model.eval()
    correct = 0
    total = 0
    
    val_running_loss = 0.0
    
    with torch.no_grad():
        for val_text, val_labels in val_loader:
            val_text = torch.stack(val_text).long()
            #print(text.shape)
            val_text = val_text.transpose(0, 1)
            val_labels = torch.as_tensor(val_labels).long()
            
            
            val_outputs = model(val_text)
            
            
            #2. calculate the loss for this batch
            batch_loss = criterion(val_outputs,val_labels)
            val_running_loss += batch_loss.item()
            
            
            
            #pick the highest score as the prediction
            _,predicted = torch.max(val_outputs.data,1)
            total += val_labels.size(0)
            correct += (predicted == val_labels).sum().item()
            
   
        
    accuracy = 100 * correct / total
    avg_val_loss  = val_running_loss / len(val_loader)
    schedular.step(avg_val_loss)
    print(f"Epoch {epoch+1}/{epochs} | Val Loss: {avg_val_loss:.4f} | Val Accuracy: {accuracy:.2f}%")
    #if (epoch + 1) % 10 == 0:
    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {epoch_loss:.4f} | Train Accuracy: {train_accuracy:.2f}%")

    #print(f"Running Loss : {epoch_loss:.4f}")
    #print(f"Epoch [{epoch  + 1}/{epochs}],Loss: {loss.item():.4f}")
    
    if avg_val_loss  < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(),'./../checkpoints/sentiment_model.pth')
    else:
        patience_counter += 1
        if patience_counter >= early_stop_patience:
            print(f"Early stopping triggered at epoch {epoch + 1}")
            break
        
        

DEBUG: Input shape at start of forward: torch.Size([32, 250])
DEBUG: Input shape at start of forward: torch.Size([32, 250])
DEBUG: Input shape at start of forward: torch.Size([32, 250])
DEBUG: Input shape at start of forward: torch.Size([32, 250])
DEBUG: Input shape at start of forward: torch.Size([32, 250])
DEBUG: Input shape at start of forward: torch.Size([32, 250])
DEBUG: Input shape at start of forward: torch.Size([32, 250])
DEBUG: Input shape at start of forward: torch.Size([32, 250])
DEBUG: Input shape at start of forward: torch.Size([32, 250])
DEBUG: Input shape at start of forward: torch.Size([32, 250])
DEBUG: Input shape at start of forward: torch.Size([32, 250])
DEBUG: Input shape at start of forward: torch.Size([32, 250])
DEBUG: Input shape at start of forward: torch.Size([32, 250])
DEBUG: Input shape at start of forward: torch.Size([32, 250])
DEBUG: Input shape at start of forward: torch.Size([32, 250])
DEBUG: Input shape at start of forward: torch.Size([32, 250])
DEBUG: I

KeyboardInterrupt: 